In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
BRONZE_PATH = "abfss://bronze@pravdatalake.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"

In [0]:
directors_df = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(f"{BRONZE_PATH}/netflix_directors")

In [0]:
directors_df.display()

In [0]:
silver_directors = (
    directors_df
    .withColumn("show_id", trim(col("show_id")))
    .withColumn("director", trim(col("director")))
    .withColumn(
        "director",
        when(col("director") == "", None)
         .otherwise(col("director"))
    )
    .filter(col("show_id").isNotNull())
    .filter(col("director").isNotNull())
    .dropDuplicates(["show_id", "director"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)



In [0]:
silver_directors.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{SILVER_PATH}/netflix_directors")